## Imports

In [1]:
import os
import pandas as pd
import numpy as np
import time
from dotenv import load_dotenv
import azure.cognitiveservices.speech as speechsdk
import csv
import seaborn as sns
import matplotlib.pyplot as plt
from tempfile import NamedTemporaryFile
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential
from azure.ai.textanalytics import HealthcareEntityRelation

In [2]:
load_dotenv() 
AZURE_SPEECH_KEY = os.getenv("SPEECHSDK_API_KEY")
AZURE_SERVICE_REGION = os.getenv("SPEECHSDK_REGION")
text_analytics_key = os.getenv("text_analytics_key")
text_analytics_endpoint = os.getenv("text_analytics_endpoint")

## Periodic processing of transcript
To achieve near real-time HER for checkbox ticking. <hr>

Checkbox options to include:
- SymptomOrSign
- Diagnosis
- BodyStructure
- Time
- TreatmentName

Optimal threshold: 0.89

In [ ]:
# optimised version

# Initialize Speech Service
def initialize_speech_service(audio_file_path):
    speech_config = speechsdk.SpeechConfig(subscription=AZURE_SPEECH_KEY, region=AZURE_SERVICE_REGION)
    audio_config = speechsdk.audio.AudioConfig(filename=audio_file_path)
    return speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_config)

# Define boolean flags and initialize tracking list
flags = {
    'SymptomOrSign': False,
    'Diagnosis': False,
    'BodyStructure': False,
    'Time': False,
    'TreatmentName': False
}
triggered_entities = []  # List to store entities that triggered flags

# Initialize Text Analytics Client outside of the handler
text_analytics_client = TextAnalyticsClient(
    endpoint=text_analytics_endpoint,
    credential=AzureKeyCredential(text_analytics_key),
)

# Perform transcription on the audio file
def transcribe_audio_with_chunk_processing(recognizer, chunk_duration=30):
    recognized_speech = []
    done = False
    last_chunk_time = time.time()
    words_to_remove = {'ok', 'yeah', 'yea', 'ya', 'hello', 'hi', 'bye', 'oh'}

    def handle_recognized(evt):
        nonlocal last_chunk_time
        recognized_speech.append(evt.result.text)

        elapsed_time = time.time() - last_chunk_time
        if elapsed_time >= chunk_duration:
            chunk_text = " ".join(recognized_speech)
            print(f"Processing 30-second chunk:\n{chunk_text}")

            hypothesis_text_list = [sent.strip() for sent in chunk_text.lower().split('. ') 
                                    if sent.strip().lower() not in words_to_remove]
            long_sentences = " ".join([sent for sent in hypothesis_text_list if len(sent.split()) > 10])

            poller = text_analytics_client.begin_analyze_healthcare_entities([long_sentences])
            result = poller.result()
            docs = [doc for doc in result if not doc.is_error]

            # Extract and create DataFrame in one step
            entities_data = [(entity.category, entity.confidence_score, entity.text) 
                             for doc in docs for entity in doc.entities if entity.category]
            her_df = pd.DataFrame(entities_data, columns=['category', 'confidence_score', 'text'])

            # Set flags based on entity categories and confidence scores and store triggering entities
            for category in flags.keys():
                if not flags[category]:  # Only check if flag is not already set
                    matches = her_df.loc[(her_df['category'] == category) & 
                                         (her_df['confidence_score'] >= 0.89)]
                    if not matches.empty:
                        flags[category] = True
                        # Save triggering entities to tracking list
                        triggered_entities.extend(matches[['category', 'confidence_score', 'text']].to_dict('records'))
            
            print("Flags status for this chunk:")
            print(flags)
            print('='*50)

            # Reset recognized speech and update time
            recognized_speech.clear()
            last_chunk_time = time.time()

    def handle_canceled(evt):
        print(f"Recognition canceled: {evt.result.reason}")
        if evt.result.reason == speechsdk.CancellationReason.Error:
            print(f"Error details: {evt.result.error_details}")
        recognizer.stop_continuous_recognition()
        nonlocal done
        done = True

    recognizer.recognized.connect(handle_recognized)
    recognizer.canceled.connect(handle_canceled)
    
    recognizer.start_continuous_recognition()
    print("Transcribing...")

    try:
        while not done:
            time.sleep(0.2)
    except KeyboardInterrupt:
        print("Transcription stopped by user.")
        recognizer.stop_continuous_recognition()

    # Save the triggered entities to a CSV after processing completes
    triggered_df = pd.DataFrame(triggered_entities)
    # triggered_df.to_csv("files/triggered_entities.csv", index=False)

    print('Overall flags:')
    return flags


In [18]:
recognizer = initialize_speech_service(r"C:\Users\zandr\OneDrive\Desktop\NUS Work\BT4103\faith-speechsdk\speechsdk\data\primock\day1_consultation01_doctor.wav") 
hypothesis_text = transcribe_audio_with_chunk_processing(recognizer, 30)

Transcribing...
Processing 30-second chunk:
Hello. Hi. Yeah. OK. Hello. Good morning. So how can I help you this morning? Yeah, I'm sorry to hear that. And when you say diarrhoea, what do you mean by diarrhea? Do you mean you're going to the toilet more often or are your stools more loose? OK. And how many times a day are you going, let's say over the last couple of days? 6-7 times a day and you mentioned this mainly water tree. Have you noticed any other things like blood in your stools? OK. And you mentioned you've had some pain in your tummy as well. Whereabouts is the pain exactly?
Flags status for this chunk:
{'SymptomOrSign': True, 'Diagnosis': False, 'BodyStructure': True, 'Time': True, 'TreatmentName': False}
Processing 30-second chunk:
One side. And what side is that? That's right. OK. And can you describe the pain to me? OK. And there's a pain. Is that is it there all the time or does it come and go? Does the pain move anywhere else because on between your back? OK, fine. And